# Geometry-conditioned language mapping from exported iPhone LiDAR

This notebook repairs the original geometry and evaluates held-out consistency before generating maps. The research direction is a hypothesis, not a novelty or state-of-the-art claim. Ground-truth metric depth, timestamps, intrinsics and confidence maps are absent. See `research/RESEARCH_REVIEW.md` for the full review and related work.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
assert (ROOT / "dataset/content/vlmaps_dataset/my_custom_scene/poses.txt").exists()
sys.path.insert(0, str(ROOT / ".packages"))
sys.path.insert(0, str(ROOT / "research"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / ".yolo")
import numpy as np
import pandas as pd
from IPython.display import display, Image, Video
import geometry_probe as probe
import improve_mapping as mapping
import verify_semantics as semantic
import query_map as language
print("Workspace:", ROOT)

## Validate inputs and evidence provenance

The user reports iPhone LiDAR. Native ARKit depth is metric, but these files appear quantized. We infer decoding in pose translation units and preserve that uncertainty.

In [ ]:
DATA = probe.DATA
poses = np.loadtxt(DATA / "poses.txt")
rgb_files = sorted((DATA / "rgb").glob("*.png"))
depth_files = sorted((DATA / "depth").glob("*.npy"))
assert len(rgb_files) == len(depth_files) == len(poses) == 2091
assert [p.stem for p in rgb_files] == [p.stem for p in depth_files]
assert poses.shape[1] == 7 and np.isfinite(poses).all()
print("Frames:", len(poses))
print("Sample raw depth:", np.percentile(np.load(depth_files[0]), [0, 1, 50, 99, 100]))
display(Image(filename=str(ROOT / "research/input_contact_sheet.png")))

## Recover camera convention and focal from training correspondences

Compare camera-to-world/world-to-camera and OpenCV/ARKit axes. Hold out every fifth anchor pair; infer only a scalar focal while retaining the centre assumption.

In [ ]:
probe.run()
diagnostics = json.loads((ROOT / "research/geometry_diagnostics.json").read_text())
print("Selected mode:", diagnostics["selected_mode"])
display(pd.DataFrame(diagnostics["camera_candidates"]).head(8))

## Recover depth and evaluate before filtering

Positive monotonic decoder spans prevent underconstrained frames from reversing depth. Pair holdout excludes duplicate image keypoints. Temporal-block depth evaluation and unfiltered 3D disagreement expose failure tails; these metrics are consistency proxies, not ground-truth accuracy.

In [ ]:
model, Khalf, mode, anchors, metrics = mapping.calibrate()
display(pd.DataFrame(metrics["depth_holdout"]))
display(pd.DataFrame(metrics["geometry_holdout"]).T)
display(pd.DataFrame(metrics["unfiltered_correspondence_fusion_holdout"]).T)
print("Temporal block depth test:", metrics["temporal_block_depth_holdout"])
display(Image(filename=str(mapping.OUT / "calibration_evaluation.png")))

## Map supported surfaces and preserve unknown space

Translation/rotation keyframes, cross-view consistency, one vote per view per voxel, and calibration-quality weighting replace raw density. Ray evidence is sampled near camera height, so this is an exploratory map, not a complete navigation costmap.

In [ ]:
points, colors, selected, Kdepth, R, t, frames, rays = mapping.mapping(model, Khalf, mode, poses)
map_image, extent = mapping.ray_map(points, selected, t, rays)
mapping.comparison(points, colors, poses)
display(Image(filename=str(mapping.OUT / "before_after_height_slice.png")))
display(Image(filename=str(mapping.OUT / "evidence_map.png")))

## Verify semantic hypotheses with text-image evidence

YOLO supplies proposals; CLIP tests an indoor vocabulary and can return unresolved. Cosine scores are not calibrated probabilities. Position support and class correctness remain separate.

In [ ]:
mapping.semantic_landmarks(model, Kdepth, R, t, selected, frames, map_image, extent)
semantic.main()
mapping.semantic_landmarks(model, Kdepth, R, t, selected, frames, map_image, extent)
display(pd.read_csv(mapping.OUT / "semantic_landmarks.csv"))
display(Image(filename=str(mapping.OUT / "semantic_evidence_map.png")))

## Sparse language retrieval with abstention

Use pooled independent-view CLIP region embeddings. This does not provide dense VLMaps semantics or guarantee proposal recall. A candidate can still have tentative spatial support.

In [ ]:
import torch
torch.set_num_threads(4)
records, embeddings = language.build_index()
clip = semantic.CLIPModel.from_pretrained(ROOT / ".models/clip", local_files_only=True).eval()
processor = semantic.CLIPProcessor.from_pretrained(ROOT / ".models/clip", local_files_only=True)
queries = ["an orange beanbag chair", "a black leather sofa", "a refrigerator"]
query_results = [language.query(q, clip, processor, records, embeddings) for q in queries]
mapping.dump("language_queries.json", query_results)
print(json.dumps(query_results, indent=2))

## VLMM-inspired motion attributes and latest research

Based on [VLMM v2 (August 2026)](https://arxiv.org/abs/2607.16173v2), with decoder-quality-aware motion covariance as the candidate extension. Related work: [Memory for Attention v2 (September)](https://arxiv.org/abs/2607.23797v2), [SuperMap (RSS 2026)](https://arxiv.org/abs/2608.22896), [ChronoGraph v2 (October)](https://arxiv.org/abs/2609.39665v2). The CPU implementation uses DIS flow and box regions; it is not a benchmark reproduction. Movability is separate from observed motion; lack of ground truth prevents motion accuracy claims.

In [ ]:
import motion_map
motion_map.main()
elements = json.loads((mapping.OUT / "vlmm_elements.json").read_text())
display(pd.DataFrame(elements)[["class_name", "observed_motion_score", "movability_prior_score", "motion_class", "eligible_motion_pairs"]])
print(json.dumps(json.loads((mapping.OUT / "vlmm_queries.json").read_text()), indent=2))
display(Image(filename=str(mapping.OUT / "vlmm_motion_attributes.png")))

## Replay and regression verification

This video uses the final map and replays the camera trajectory; it is not an online-building benchmark. Tests cover the geometric convention, calibration recovery, monotonicity and evidence double counting.

In [ ]:
mapping.render_replay(poses)
display(Video(filename=str(mapping.OUT / "repaired_room_replay.mp4"), embed=True))
import unittest, test_mapping
suite = unittest.defaultTestLoader.loadTestsFromModule(test_mapping)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print("Completed: saved map, language evidence, holdout metrics, replay and passing regression tests.")

## Research interpretation

The strongest current result is recovering a coherent room from incorrectly interpreted exported data. Much of the gain comes from fixing axes and calibration, which is engineering, not novelty. The proposed research question is whether decoder-quality-conditioned evidence budgeting improves language localization and abstention on lossy mobile LiDAR exports across scenes. The unfiltered tail remains worse than the camera-only intermediate method; filtering trades completeness for support. Use scene-level splits and metric/semantic ground truth before publication claims.

References: [VLMaps](https://arxiv.org/abs/2210.05714), [Affine Corrections](https://arxiv.org/abs/2501.05446), [Semantic Mapping Overconfidence](https://motion.cs.illinois.edu/papers/ICRA2024_Marques_OverconfidenceSemanticMapping.pdf), [SCOUT](https://arxiv.org/abs/2606.06721).

The stronger, narrower candidate is export-calibration-aware motion grounding. It needs dynamic scenes with known ground truth; the current sequence does not establish uniqueness or motion recall.